In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_proposed_restructuring_fldt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ProposedRestructuringId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "ProposedRestructuringKey string,"
        #     "ProposedRestructuringProductKey string,"
        #     "ExposureKey string,"
        #     "AgreementKey string,"
        #     "OriginProductKey string,"
        #     "ProductKey string,"
        #     "ProductCode string,"
        #     "Product string,"
        #     "ProductCategory int,"
        #     "ProductCategoryType string,"
        #     "Currency string,"
        #     "ExchangeRate string,"
        #     "Priority int,"
        #     "ProductTypePriority int,"
        #     "NPV decimal(18,6),"
        #     "BalanceForNPVCalculation decimal(18,6),"
        #     "DownPaymentAmount decimal(18,6),"
        #     "BalanceForRestructuring decimal(18,6),"
        #     "ExpensesForRestructuring decimal(18,6),"
        #     "HasStepUpFlag boolean,"
        #     "HasHaircutFlag boolean,"
        #     "HaircutPercentage decimal(18,6),"
        #     "HaircutPercentageOnBalance decimal(18,6),"
        #     "HaircutPercentageOffBalance decimal(18,6),"
        #     "HaircutAmount decimal(18,6),"
        #     "HaircutAmountOnBalance decimal(18,6),"
        #     "HaircutAmountOffBalance decimal(18,6),"
        #     "HasHaircutWithClawBackFlag int,"
        #     "HasSplitBalanceFlag boolean,"
        #     "SplitBalancePercentage decimal(18,6),"
        #     "AutoCreditStatusCode int,"
        #     "AutoCreditStatus string,"
        #     "HasNonAutoCreditReasonFlag boolean,"
        #     "NonAutoCreditReasonVisibleFlag boolean,"
        #     "IsViableFlag boolean,"
        #     "InstallmentAmount decimal(18,6),"
        #     "TotalinterestRate decimal(18,6),"
        #     "BaseRate string,"
        #     "SpreadRate decimal(18,6),"
        #     "Law128Rate decimal(18,6),"
        #     "PenaltyinterestRate decimal(18,6),"
        #     "FirstPeriodTotalinterestRate decimal(18,6),"
        #     "FirstPeriodFixedRate decimal(18,6),"
        #     "FirstPeriodBaseRate string,"
        #     "FirstPeriodSpreadRate decimal(18,6),"
        #     "FirstPeriodLaw128Rate decimal(18,6),"
        #     "FirstPeriodinterestDurationInMonths int,"
        #     "RequiresNewPrenotationFlag boolean,"
        #     "DifferentinterestsDuringPeriod int,"
        #     "ReducedInstalmentPeriod int,"
        #     "ReducedInstalmentPeriodInMonths int,"
        #     "ReducedInstalmentPeriodAmount decimal(18,6),"
        #     "TotalTenureInMonths int,"
        #     "SmallTicketAmount decimal(18,6),"
        #     "IsVisibleFlag boolean,"
        #     "IsLiquidationFlag boolean,"
        #     "TenureInMonthsMin int,"
        #     "TenureInMonthsMax int,"
        #     "HaircutPercentageMin decimal(18,6),"
        #     "HaircutPercentageMax decimal(18,6),"
        #     "IsEditableFlag boolean,"
        #     "IsCustomFlag boolean,"
        #     "ProposedRestructuringAmountAfterHaircut decimal(18,6),"
        #     "StepUpPeriodTenureInMonths int,"
        #     "StepUpStartMonth int,"
        #     "StepUpReadjustmentDuration int,"
        #     "StepUpPeriodNumber int,"
        #     "StepUpInitialInstallmentAmount decimal(18,6),"
        #     "StepUpAmount decimal(18,6),"
        #     "DurationAfterStepUpExpirationInMonths int,"
        #     "SplitBalanceAmountA decimal(18,6),"
        #     "SplitBalanceAmountB decimal(18,6),"
        #     "SplitBalanceTenureInMonthsA int,"
        #     "SplitBalanceStartMonthB int,"
        #     "SplitBalanceTenureInMonthsB int,"
        #     "ProposedRestructuringNumber int,"
        #     "ProposedRestructuringIdentifier string,"
        #     "ProposedRestructuringProductCode string,"
        #     "ProposedRestructuringType int,"
        #     "ProposedRestructuringBalance decimal(18,6),"
        #     "ProposedRestructuringStartMonth int,"
        #     "ProposedRestructuringEndMonth int,"
        #     "ProposedRestructuringTenureInMonths int,"
        #     "HasReducedInstallmentPeriodFlag boolean,"
        #     "HasStepUpPeriodFlag boolean,"
        #     "StepUpPercentage decimal(18,6),"
        #     "StepUpPeriodInMonths int,"
        #     "InstallmentPeriodKey string,"
        #     "InstallmentPeriodPriority int,"
        #     "InstallmentPeriodTypeCode int,"
        #     "InstallmentPeriodType string,"
        #     "InstallmentPeriodInstalmentAmount decimal(18,6),"
        #     "InstallmentPeriodStartMonth int,"
        #     "InstallmentPeriodEndMonth int,"
        #     "InstallmentPeriodTenureInMonths int,"
        #     "InstallmentPeriodBaseinterest decimal(18,6),"
        #     "InstallmentPeriodBaseinterestTypeCode int,"
        #     "InstallmentPeriodBaseinterestType string,"
        #     "IsFixedInstallmentPeriodBaseinterestFlag int,"
        #     "InstallmentPeriodinterestCapitalisationInMonths int,"
        #     "InstallmentPeriodinterestSpreadRate decimal(18,6),"
        #     "InstallmentPeriodLaw128Rate decimal(18,6),"
        #     "InstallmentPeriodTotalinterestRate decimal(18,6),"
        #     "AutoCreditReasonKey string,"
        #     "AutoCreditReasonCode string,"
        #     "AutoCreditReason string,"
        #     "AutoCreditReasonCategoryCode string,"
        #     "AutoCreditReasonCategory string,"
        #     "FinalRestructuringKey string,"
        #     "FinalRestructuringProductKey string,"
        #     "FinalRestructuringProductComment string,"
        #     "AssetClass int,"
        #     "ProposedEditReasonCode string,"
        #     "ProposedEditReason string,"
        #     "DateOfFirstInstallment timestamp,"
        #     "FinalRestructuringStatusCode int,"
        #     "FinalRestructuringStatus string,"
        #     "IsFinalRestructuringActiveFlag boolean,"
        #     "FinalRestructuringRejectReasonCode string,"
        #     "FinalRestructuringRejectReason string,"
        #     "FinalRestructuringComment string,"
        #     "FinalRestructuringAutoCreditStatusCode int,"
        #     "FinalRestructuringCreditStatus string,"
        #     "CreatedDate timestamp,"
        #     "CreatedBy string,"
        #     "CreatedByName string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ProposedRestructuring"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "fldt",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== suggestedmodificationproduct ==================

        suggestedmodificationproduct_extra_filters = None

        if suggestedmodificationproduct_extra_filters is not None:
            suggestedmodificationproduct_extra_filters.append(date_filter)
            suggestedmodificationproduct_final_filters = " AND ".join(
                suggestedmodificationproduct_extra_filters
            )
        else:
            suggestedmodificationproduct_final_filters = date_filter

        suggestedmodificationproduct_selected_fields = [
            "id",
            "originproductkey",
            "productkey",
            "code",
            "name",
            "productcategory",
            "productcategorytype",
            "modificationcurrency",
            "priority",
            "producttypepriority",
            "npv",
            "balanceusedfornpvcalculation",
            "downpaymentamount",
            "balanceformodification",
            "expensesofmodificationamount",
            "hasstepup",
            "hashaircut",
            "haircutpercentage",
            "haircutpercentageonbalance",
            "haircutpercentagewithoffbalance",
            "haircutamount",
            "haircutamountonbalance",
            "haircutamountwithoffbalance",
            "hashaircutwithclawback",
            "hassplitbalance",
            "splitbalancepercentage",
            "autocreditstatus",
            "autocreditstatusdescription",
            "hasnonautocreditreasons",
            "nonautocreditreasonsvisible",
            "isviable",
            "installmentamountcalculated",
            "totalinterestrate",
            "baseinterest",
            "spread",
            "law128ratevalue",
            "penaltyinterestrate",
            "firstperiodtotalinterestrate",
            "fixedinterestoffirstperiod",
            "baseinteresttypeoffirstperiod",
            "spreadoffirstperiod",
            "law128rateoffirstperiodvalue",
            "interestdurationoffirstperiodinmonths",
            "requiresnewprenotation",
            "differentinterestsduringperiods",
            "reducedinstalmentperiod",
            "reducedinstalmentperiodinmonths",
            "reducedinstalmentperiodamount",
            "totaltenureinmonths",
            "smallticketamount",
            "isvisible",
            "isliquidation",
            "tenureinmonthsmin",
            "tenureinmonthsmax",
            "haircutpercentagemin",
            "haircutpercentagemax",
            "iseditable",
            "iscustom",
            "modificationamountafterhaircut",
            "stepupperiodtenureinmonths",
            "stepupstartmonth",
            "stepupreadjustmentduration",
            "stepupperiodnumber",
            "stepupinitialinstallmentamount",
            "stepupamount",
            "durationafterstepupexpirationinmonths",
            "splitbalanceamounta",
            "splitbalanceamountb",
            "splitbalancetenureinmonthsa",
            "splitbalancestartmonthb",
            "splitbalancetenureinmonthsb",
            "loanid",
            "businessdate",
        ]

        suggestedmodificationproduct = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_suggestedmodificationproduct",
            suggestedmodificationproduct_final_filters,
            suggestedmodificationproduct_selected_fields,
        )

        # ================== loan ==================

        loan_extra_filters = None

        if loan_extra_filters is not None:
            loan_extra_filters.append(date_filter)
            loan_final_filters = " AND ".join(loan_extra_filters)
        else:
            loan_final_filters = date_filter

        loan_selected_fields = [
            "contractnumbersystemicnr",
            "id",
            "contractnumber",
            "accountnumberwithprefix",
            "businessdate",
        ]

        loan = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_loan",
            loan_final_filters,
            loan_selected_fields,
        )

        # ================== unifiedloan ==================

        unifiedloan_extra_filters = None

        if unifiedloan_extra_filters is not None:
            unifiedloan_extra_filters.append(date_filter)
            unifiedloan_final_filters = " AND ".join(unifiedloan_extra_filters)
        else:
            unifiedloan_final_filters = date_filter

        unifiedloan_selected_fields = [
            "accountnumber",
            "accountnumberwithprefix",
            "loanid",
            "businessdate",
        ]

        unifiedloan = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_unifiedloan",
            unifiedloan_final_filters,
            unifiedloan_selected_fields,
        )

        # ================== suggestedmodificationproductpart ==================

        suggestedmodificationproductpart_extra_filters = None

        if suggestedmodificationproductpart_extra_filters is not None:
            suggestedmodificationproductpart_extra_filters.append(date_filter)
            suggestedmodificationproductpart_final_filters = " AND ".join(
                suggestedmodificationproductpart_extra_filters
            )
        else:
            suggestedmodificationproductpart_final_filters = date_filter

        suggestedmodificationproductpart_selected_fields = [
            "id",
            "partnumber",
            "partidentifier",
            "partproductcode",
            "parttype",
            "partbalance",
            "partstartmonth",
            "partendmonth",
            "parttenureinmonths",
            "parthasreducedinstallmentperiod",
            "parthasstepupperiod",
            "partstepuppercentage",
            "partstepupperiodinmonths",
            "suggestedmodificationproductid",
            "businessdate",
        ]

        suggestedmodificationproductpart = (
            load_landed_delta_table(  # noqa: F841
                spark,
                catalog_landed,  # type: ignore
                "fldt_suggestedmodificationproductpart",
                suggestedmodificationproductpart_final_filters,
                suggestedmodificationproductpart_selected_fields,
            )
        )

        # ================== installmentperiod ==================

        installmentperiod_extra_filters = None

        if installmentperiod_extra_filters is not None:
            installmentperiod_extra_filters.append(date_filter)
            installmentperiod_final_filters = " AND ".join(
                installmentperiod_extra_filters
            )
        else:
            installmentperiod_final_filters = date_filter

        installmentperiod_selected_fields = [
            "id",
            "periodpriority",
            "periodtype",
            "periodtypedescription",
            "periodinstalmentamount",
            "periodstartmonth",
            "periodendmonth",
            "periodtenureinmonths",
            "periodbaseinterest",
            "periodbaseinteresttype",
            "baseinteresttypedescription",
            "periodbaseinterestisfixed",
            "periodinterestcapitalisationinmonths",
            "periodinterestspread",
            "periodlaw128rate",
            "periodtotalinterestrate",
            "suggestedmodificationproductpartid",
            "businessdate",
        ]

        installmentperiod = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "fldt_installmentperiod",
            installmentperiod_final_filters,
            installmentperiod_selected_fields,
        )

        # ================== suggestedmodificationproductautocreditreason ==================

        suggestedmodificationproductautocreditreason_extra_filters = None

        if suggestedmodificationproductautocreditreason_extra_filters is not None:
            suggestedmodificationproductautocreditreason_extra_filters.append(
                date_filter
            )
            suggestedmodificationproductautocreditreason_final_filters = " AND ".join(
                suggestedmodificationproductautocreditreason_extra_filters
            )
        else:
            suggestedmodificationproductautocreditreason_final_filters = date_filter

        suggestedmodificationproductautocreditreason_selected_fields = [
            "id",
            "autocreditreasoncode",
            "autocreditreasondescription",
            "autocreditreasoncategorycode",
            "autocreditreasoncategorydescription",
            "suggestedmodificationproductid",
            "businessdate",
        ]

        suggestedmodificationproductautocreditreason = (
            load_landed_delta_table(  # noqa: F841
                spark,
                catalog_landed,  # type: ignore
                "fldt_suggestedmodificationproductautocreditreason",
                suggestedmodificationproductautocreditreason_final_filters,
                suggestedmodificationproductautocreditreason_selected_fields,
            )
        )

        # ================== finalsolution ==================

        finalsolution_extra_filters = None

        if finalsolution_extra_filters is not None:
            finalsolution_extra_filters.append(date_filter)
            finalsolution_final_filters = " AND ".join(finalsolution_extra_filters)
        else:
            finalsolution_final_filters = date_filter

        finalsolution_selected_fields = [
            "id",
            "finalsolutionstatus",
            "finalsolutionstatusdescription",
            "isactive",
            "rejectcode",
            "rejectcodedescription",
            "solutioncomment",
            "resultautocreditstatus",
            "resultautocreditstatusdescription",
            "username",
            "fullname",
            "dateinserted",
            "businessdate",
        ]

        finalsolution = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "fldt_finalsolution",
            finalsolution_final_filters,
            finalsolution_selected_fields,
        )

        # ================== finalmodificationproduct ==================

        finalmodificationproduct_extra_filters = None

        if finalmodificationproduct_extra_filters is not None:
            finalmodificationproduct_extra_filters.append(date_filter)
            finalmodificationproduct_final_filters = " AND ".join(
                finalmodificationproduct_extra_filters
            )
        else:
            finalmodificationproduct_final_filters = date_filter

        finalmodificationproduct_selected_fields = [
            "id",
            "finalsolutionid",
            "comment",
            "assetclass",
            "suggestioneditreasoncode",
            "suggestioneditreasondescription",
            "dateoffirstinstallment",
            "productkey",
            "loanid",
            "businessdate",
        ]

        finalmodificationproduct = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "fldt_finalmodificationproduct",
            finalmodificationproduct_final_filters,
            finalmodificationproduct_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "portfolioname",
            "prefix",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # Join conditions
        suggestedmodificationproduct_finalmodificationproduct_join_condition = (
            suggestedmodificationproduct["productkey"]
            == finalmodificationproduct["productkey"]
        ) & (
            suggestedmodificationproduct["loanid"] == finalmodificationproduct["loanid"]
        )  # full outer join

        finalmodificationproduct_finalsolution_join_condition = (
            finalmodificationproduct["finalsolutionid"] == finalsolution["id"]
        )  # left

        suggestedmodificationproduct_finalmodificationproduct_loan_join_condition = (
            coalesce(
                suggestedmodificationproduct["loanid"],
                finalmodificationproduct["loanid"],
            )
            == loan["id"]
        )  # left

        loan_unifiedloan_join_condition = loan["id"] == unifiedloan["loanid"]  # left

        suggestedmodificationproduct_suggestedmodificationproductpart_join_condition = (
            suggestedmodificationproduct["id"]
            == suggestedmodificationproductpart["suggestedmodificationproductid"]
        )  # left

        suggestedmodificationproductpart_installmentperiod_join_condition = (
            suggestedmodificationproductpart["id"]
            == installmentperiod["suggestedmodificationproductpartid"]
        )  # left

        suggestedmodificationproductpart_suggestedmodificationproductautocreditreason_join_condition = (
            suggestedmodificationproductpart["suggestedmodificationproductid"]
            == suggestedmodificationproductautocreditreason[
                "suggestedmodificationproductid"
            ]
        )  # left

        loan_portfolio_join_condition = (
            coalesce(
                substring(loan["contractnumbersystemicnr"], 1, 3),
                substring(loan["contractnumber"], 1, 3),
            )
            == portfolio["prefix"]
        )  # left

        # Joins
        target_table_fldt = (
            suggestedmodificationproduct.join(
                finalmodificationproduct,
                suggestedmodificationproduct_finalmodificationproduct_join_condition,
                "full_outer",
            )
            .join(
                finalsolution,
                finalmodificationproduct_finalsolution_join_condition,
                "left",
            )
            .join(
                loan,
                suggestedmodificationproduct_finalmodificationproduct_loan_join_condition,
                "left",
            )
            .join(unifiedloan, loan_unifiedloan_join_condition, "left")
            .join(
                suggestedmodificationproductpart,
                suggestedmodificationproduct_suggestedmodificationproductpart_join_condition,
                "left",
            )
            .join(
                installmentperiod,
                suggestedmodificationproductpart_installmentperiod_join_condition,
                "left",
            )
            .join(
                suggestedmodificationproductautocreditreason,
                suggestedmodificationproductpart_suggestedmodificationproductautocreditreason_join_condition,
                "left",
            )
            .join(portfolio, loan_portfolio_join_condition, "left")
        )

        # HashKeys
        target_table_fldt_ProposedRestructuringId_primary_key_fields = [
            suggestedmodificationproduct.id,
            finalmodificationproduct.id,
            coalesce(loan.accountnumberwithprefix, unifiedloan.accountnumberwithprefix),
            installmentperiod.id,
            suggestedmodificationproductautocreditreason.id,
            lit("fldt"),
        ]

        # Target table(hash keys and calculations)
        target_table_fldt = target_table_fldt.withColumn(
            "ProposedRestructuringId",
            sha2(
                concat_ws(
                    ";", *target_table_fldt_ProposedRestructuringId_primary_key_fields
                ),
                256,
            ),
        ).withColumn(
            "ProposedRestructuringKey",
            concat_ws(
                ";",
                suggestedmodificationproduct.id,
                finalmodificationproduct.id,
                coalesce(
                    loan.accountnumberwithprefix, unifiedloan.accountnumberwithprefix
                ),
                installmentperiod.id,
                suggestedmodificationproductautocreditreason.id,
            ),
        )

        # ================== Selection of Final Columns ==================

        result_df_fldt = target_table_fldt.select(
            target_table_fldt.ProposedRestructuringId.alias("ProposedRestructuringId"),
            lit("fldt").cast("string").alias("SourceSystem"),
            substring(loan.contractnumbersystemicnr, 1, 3)
            .cast("string")
            .alias("PortfolioKey"),
            portfolio.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            target_table_fldt.ProposedRestructuringKey.cast("string").alias(
                "ProposedRestructuringKey"
            ),
            suggestedmodificationproduct.id.cast("string").alias(
                "ProposedRestructuringProductKey"
            ),
            coalesce(loan.accountnumberwithprefix, unifiedloan.accountnumberwithprefix)
            .cast("string")
            .alias("ExposureKey"),
            loan.contractnumbersystemicnr.cast("string").alias("AgreementKey"),
            suggestedmodificationproduct.originproductkey.cast("string").alias(
                "OriginProductKey"
            ),
            coalesce(
                suggestedmodificationproduct.productkey,
                finalmodificationproduct.productkey,
            )
            .cast("string")
            .alias("ProductKey"),
            suggestedmodificationproduct.code.cast("string").alias("ProductCode"),
            suggestedmodificationproduct.name.cast("string").alias("Product"),
            suggestedmodificationproduct.productcategory.cast("int").alias(
                "ProductCategory"
            ),
            suggestedmodificationproduct.productcategorytype.cast("string").alias(
                "ProductCategoryType"
            ),
            suggestedmodificationproduct.modificationcurrency.cast("string").alias(
                "Currency"
            ),
            lit(None).cast("string").alias("ExchangeRate"),
            suggestedmodificationproduct.priority.cast("int").alias("Priority"),
            suggestedmodificationproduct.producttypepriority.cast("int").alias(
                "ProductTypePriority"
            ),
            suggestedmodificationproduct.npv.cast("decimal(18,6)").alias("NPV"),
            suggestedmodificationproduct.balanceusedfornpvcalculation.cast(
                "decimal(18,6)"
            ).alias("BalanceForNPVCalculation"),
            suggestedmodificationproduct.downpaymentamount.cast("decimal(18,6)").alias(
                "DownPaymentAmount"
            ),
            suggestedmodificationproduct.balanceformodification.cast(
                "decimal(18,6)"
            ).alias("BalanceForRestructuring"),
            suggestedmodificationproduct.expensesofmodificationamount.cast(
                "decimal(18,6)"
            ).alias("ExpensesForRestructuring"),
            suggestedmodificationproduct.hasstepup.cast("boolean").alias(
                "HasStepUpFlag"
            ),
            suggestedmodificationproduct.hashaircut.cast("boolean").alias(
                "HasHaircutFlag"
            ),
            suggestedmodificationproduct.haircutpercentage.cast("decimal(18,6)").alias(
                "HaircutPercentage"
            ),
            suggestedmodificationproduct.haircutpercentageonbalance.cast(
                "decimal(18,6)"
            ).alias("HaircutPercentageOnBalance"),
            suggestedmodificationproduct.haircutpercentagewithoffbalance.cast(
                "decimal(18,6)"
            ).alias("HaircutPercentageOffBalance"),
            suggestedmodificationproduct.haircutamount.cast("decimal(18,6)").alias(
                "HaircutAmount"
            ),
            suggestedmodificationproduct.haircutamountonbalance.cast(
                "decimal(18,6)"
            ).alias("HaircutAmountOnBalance"),
            suggestedmodificationproduct.haircutamountwithoffbalance.cast(
                "decimal(18,6)"
            ).alias("HaircutAmountOffBalance"),
            suggestedmodificationproduct.hashaircutwithclawback.cast("int").alias(
                "HasHaircutWithClawBackFlag"
            ),
            suggestedmodificationproduct.hassplitbalance.cast("boolean").alias(
                "HasSplitBalanceFlag"
            ),
            suggestedmodificationproduct.splitbalancepercentage.cast(
                "decimal(18,6)"
            ).alias("SplitBalancePercentage"),
            suggestedmodificationproduct.autocreditstatus.cast("int").alias(
                "AutoCreditStatusCode"
            ),
            suggestedmodificationproduct.autocreditstatusdescription.cast(
                "string"
            ).alias("AutoCreditStatus"),
            suggestedmodificationproduct.hasnonautocreditreasons.cast("boolean").alias(
                "HasNonAutoCreditReasonFlag"
            ),
            suggestedmodificationproduct.nonautocreditreasonsvisible.cast(
                "boolean"
            ).alias("NonAutoCreditReasonVisibleFlag"),
            suggestedmodificationproduct.isviable.cast("boolean").alias("IsViableFlag"),
            suggestedmodificationproduct.installmentamountcalculated.cast(
                "decimal(18,6)"
            ).alias("InstallmentAmount"),
            suggestedmodificationproduct.totalinterestrate.cast("decimal(18,6)").alias(
                "TotalInterestRate"
            ),
            suggestedmodificationproduct.baseinterest.cast("string").alias("BaseRate"),
            suggestedmodificationproduct.spread.cast("decimal(18,6)").alias(
                "SpreadRate"
            ),
            suggestedmodificationproduct.law128ratevalue.cast("decimal(18,6)").alias(
                "Law128Rate"
            ),
            suggestedmodificationproduct.penaltyinterestrate.cast(
                "decimal(18,6)"
            ).alias("PenaltyInterestRate"),
            suggestedmodificationproduct.firstperiodtotalinterestrate.cast(
                "decimal(18,6)"
            ).alias("FirstPeriodTotalInterestRate"),
            suggestedmodificationproduct.fixedinterestoffirstperiod.cast(
                "decimal(18,6)"
            ).alias("FirstPeriodFixedRate"),
            suggestedmodificationproduct.baseinteresttypeoffirstperiod.cast(
                "string"
            ).alias("FirstPeriodBaseRate"),
            suggestedmodificationproduct.spreadoffirstperiod.cast(
                "decimal(18,6)"
            ).alias("FirstPeriodSpreadRate"),
            suggestedmodificationproduct.law128rateoffirstperiodvalue.cast(
                "decimal(18,6)"
            ).alias("FirstPeriodLaw128Rate"),
            suggestedmodificationproduct.interestdurationoffirstperiodinmonths.cast(
                "int"
            ).alias("FirstPeriodInterestDurationInMonths"),
            suggestedmodificationproduct.requiresnewprenotation.cast("boolean").alias(
                "RequiresNewPrenotationFlag"
            ),
            suggestedmodificationproduct.differentinterestsduringperiods.cast(
                "int"
            ).alias("DifferentInterestsDuringPeriod"),
            suggestedmodificationproduct.reducedinstalmentperiod.cast("int").alias(
                "ReducedInstalmentPeriod"
            ),
            suggestedmodificationproduct.reducedinstalmentperiodinmonths.cast(
                "int"
            ).alias("ReducedInstalmentPeriodInMonths"),
            suggestedmodificationproduct.reducedinstalmentperiodamount.cast(
                "decimal(18,6)"
            ).alias("ReducedInstalmentPeriodAmount"),
            suggestedmodificationproduct.totaltenureinmonths.cast("int").alias(
                "TotalTenureInMonths"
            ),
            suggestedmodificationproduct.smallticketamount.cast("decimal(18,6)").alias(
                "SmallTicketAmount"
            ),
            suggestedmodificationproduct.isvisible.cast("boolean").alias(
                "IsVisibleFlag"
            ),
            suggestedmodificationproduct.isliquidation.cast("boolean").alias(
                "IsLiquidationFlag"
            ),
            suggestedmodificationproduct.tenureinmonthsmin.cast("int").alias(
                "TenureInMonthsMin"
            ),
            suggestedmodificationproduct.tenureinmonthsmax.cast("int").alias(
                "TenureInMonthsMax"
            ),
            suggestedmodificationproduct.haircutpercentagemin.cast(
                "decimal(18,6)"
            ).alias("HaircutPercentageMin"),
            suggestedmodificationproduct.haircutpercentagemax.cast(
                "decimal(18,6)"
            ).alias("HaircutPercentageMax"),
            suggestedmodificationproduct.iseditable.cast("boolean").alias(
                "IsEditableFlag"
            ),
            suggestedmodificationproduct.iscustom.cast("boolean").alias("IsCustomFlag"),
            suggestedmodificationproduct.modificationamountafterhaircut.cast(
                "decimal(18,6)"
            ).alias("ProposedRestructuringAmountAfterHaircut"),
            suggestedmodificationproduct.stepupperiodtenureinmonths.cast("int").alias(
                "StepUpPeriodTenureInMonths"
            ),
            suggestedmodificationproduct.stepupstartmonth.cast("int").alias(
                "StepUpStartMonth"
            ),
            suggestedmodificationproduct.stepupreadjustmentduration.cast("int").alias(
                "StepUpReadjustmentDuration"
            ),
            suggestedmodificationproduct.stepupperiodnumber.cast("int").alias(
                "StepUpPeriodNumber"
            ),
            suggestedmodificationproduct.stepupinitialinstallmentamount.cast(
                "decimal(18,6)"
            ).alias("StepUpInitialInstallmentAmount"),
            suggestedmodificationproduct.stepupamount.cast("decimal(18,6)").alias(
                "StepUpAmount"
            ),
            suggestedmodificationproduct.durationafterstepupexpirationinmonths.cast(
                "int"
            ).alias("DurationAfterStepUpExpirationInMonths"),
            suggestedmodificationproduct.splitbalanceamounta.cast(
                "decimal(18,6)"
            ).alias("SplitBalanceAmountA"),
            suggestedmodificationproduct.splitbalanceamountb.cast(
                "decimal(18,6)"
            ).alias("SplitBalanceAmountB"),
            suggestedmodificationproduct.splitbalancetenureinmonthsa.cast("int").alias(
                "SplitBalanceTenureInMonthsA"
            ),
            suggestedmodificationproduct.splitbalancestartmonthb.cast("int").alias(
                "SplitBalanceStartMonthB"
            ),
            suggestedmodificationproduct.splitbalancetenureinmonthsb.cast("int").alias(
                "SplitBalanceTenureInMonthsB"
            ),
            suggestedmodificationproductpart.partnumber.cast("int").alias(
                "ProposedRestructuringNumber"
            ),
            suggestedmodificationproductpart.partidentifier.cast("string").alias(
                "ProposedRestructuringIdentifier"
            ),
            suggestedmodificationproductpart.partproductcode.cast("string").alias(
                "ProposedRestructuringProductCode"
            ),
            suggestedmodificationproductpart.parttype.cast("int").alias(
                "ProposedRestructuringType"
            ),
            suggestedmodificationproductpart.partbalance.cast("decimal(18,6)").alias(
                "ProposedRestructuringBalance"
            ),
            suggestedmodificationproductpart.partstartmonth.cast("int").alias(
                "ProposedRestructuringStartMonth"
            ),
            suggestedmodificationproductpart.partendmonth.cast("int").alias(
                "ProposedRestructuringEndMonth"
            ),
            suggestedmodificationproductpart.parttenureinmonths.cast("int").alias(
                "ProposedRestructuringTenureInMonths"
            ),
            suggestedmodificationproductpart.parthasreducedinstallmentperiod.cast(
                "boolean"
            ).alias("HasReducedInstallmentPeriodFlag"),
            suggestedmodificationproductpart.parthasstepupperiod.cast("boolean").alias(
                "HasStepUpPeriodFlag"
            ),
            suggestedmodificationproductpart.partstepuppercentage.cast(
                "decimal(18,6)"
            ).alias("StepUpPercentage"),
            suggestedmodificationproductpart.partstepupperiodinmonths.cast("int").alias(
                "StepUpPeriodInMonths"
            ),
            installmentperiod.id.cast("string").alias("InstallmentPeriodKey"),
            installmentperiod.periodpriority.cast("int").alias(
                "InstallmentPeriodPriority"
            ),
            installmentperiod.periodtype.cast("int").alias("InstallmentPeriodTypeCode"),
            installmentperiod.periodtypedescription.cast("string").alias(
                "InstallmentPeriodType"
            ),
            installmentperiod.periodinstalmentamount.cast("decimal(18,6)").alias(
                "InstallmentPeriodInstalmentAmount"
            ),
            installmentperiod.periodstartmonth.cast("int").alias(
                "InstallmentPeriodStartMonth"
            ),
            installmentperiod.periodendmonth.cast("int").alias(
                "InstallmentPeriodEndMonth"
            ),
            installmentperiod.periodtenureinmonths.cast("int").alias(
                "InstallmentPeriodTenureInMonths"
            ),
            installmentperiod.periodbaseinterest.cast("decimal(18,6)").alias(
                "InstallmentPeriodBaseInterest"
            ),
            installmentperiod.periodbaseinteresttype.cast("int").alias(
                "InstallmentPeriodBaseInterestTypeCode"
            ),
            installmentperiod.baseinteresttypedescription.cast("string").alias(
                "InstallmentPeriodBaseInterestType"
            ),
            installmentperiod.periodbaseinterestisfixed.cast("int").alias(
                "IsFixedInstallmentPeriodBaseInterestFlag"
            ),
            installmentperiod.periodinterestcapitalisationinmonths.cast("int").alias(
                "InstallmentPeriodInterestCapitalisationInMonths"
            ),
            installmentperiod.periodinterestspread.cast("decimal(18,6)").alias(
                "InstallmentPeriodInterestSpreadRate"
            ),
            installmentperiod.periodlaw128rate.cast("decimal(18,6)").alias(
                "InstallmentPeriodLaw128Rate"
            ),
            installmentperiod.periodtotalinterestrate.cast("decimal(18,6)").alias(
                "InstallmentPeriodTotalInterestRate"
            ),
            suggestedmodificationproductautocreditreason.id.cast("string").alias(
                "AutoCreditReasonKey"
            ),
            suggestedmodificationproductautocreditreason.autocreditreasoncode.cast(
                "string"
            ).alias("AutoCreditReasonCode"),
            suggestedmodificationproductautocreditreason.autocreditreasondescription.cast(
                "string"
            ).alias(
                "AutoCreditReason"
            ),
            suggestedmodificationproductautocreditreason.autocreditreasoncategorycode.cast(
                "string"
            ).alias(
                "AutoCreditReasonCategoryCode"
            ),
            suggestedmodificationproductautocreditreason.autocreditreasoncategorydescription.cast(
                "string"
            ).alias(
                "AutoCreditReasonCategory"
            ),
            finalsolution.id.cast("string").alias("FinalRestructuringKey"),
            finalmodificationproduct.finalsolutionid.cast("string").alias(
                "FinalRestructuringProductKey"
            ),
            finalmodificationproduct.comment.cast("string").alias(
                "FinalRestructuringProductComment"
            ),
            finalmodificationproduct.assetclass.cast("int").alias("AssetClass"),
            finalmodificationproduct.suggestioneditreasoncode.cast("string").alias(
                "ProposedEditReasonCode"
            ),
            finalmodificationproduct.suggestioneditreasondescription.cast(
                "string"
            ).alias("ProposedEditReason"),
            finalmodificationproduct.dateoffirstinstallment.cast("timestamp").alias(
                "DateOfFirstInstallment"
            ),
            finalsolution.finalsolutionstatus.cast("int").alias(
                "FinalRestructuringStatusCode"
            ),
            finalsolution.finalsolutionstatusdescription.cast("string").alias(
                "FinalRestructuringStatus"
            ),
            finalsolution.isactive.cast("boolean").alias(
                "IsFinalRestructuringActiveFlag"
            ),
            finalsolution.rejectcode.cast("string").alias(
                "FinalRestructuringRejectReasonCode"
            ),
            finalsolution.rejectcodedescription.cast("string").alias(
                "FinalRestructuringRejectReason"
            ),
            finalsolution.solutioncomment.cast("string").alias(
                "FinalRestructuringComment"
            ),
            finalsolution.resultautocreditstatus.cast("int").alias(
                "FinalRestructuringAutoCreditStatusCode"
            ),
            finalsolution.resultautocreditstatusdescription.cast("string").alias(
                "FinalRestructuringCreditStatus"
            ),
            finalsolution.dateinserted.cast("timestamp").alias("CreatedDate"),
            finalsolution.username.cast("string").alias("CreatedBy"),
            finalsolution.fullname.cast("string").alias("CreatedByName"),
            coalesce(
                suggestedmodificationproduct.businessdate,
                finalmodificationproduct.businessdate,
            )
            .cast("date")
            .alias("BusinessDate"),
        )

        result_df_fldt.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_proposed_restructuring_fldt(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)